# Aula 2 - Context Engineering

## Versão 2.0 do projeto: um assistente que seleciona contexto

Nesta aula, a aplicação vai selecionar documentos locais antes de chamar uma LLM. Precisamos responder: "qual informação entra na requisição para responder esta pergunta?"

Ao final, você deve conseguir:

- separar instruções, pergunta, histórico e dados externos;
- selecionar documentos por uma regra simples de relevância;
- limitar o tamanho do contexto;
- montar uma requisição que identifica suas fontes;
- comparar uma resposta sem contexto com uma resposta contextualizada.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

## 2. Crie uma base local de documentos

Cada documento tem um identificador, um título, um conteúdo e termos que ajudam a explicar por que ele foi selecionado. Em um sistema real, essa base poderia vir de arquivos, banco de dados ou uma API.

In [ ]:
# TODO: crie uma lista chamada `knowledge_base` com pelo menos 5 documentos.
# Cada item deve ter: id, title, content e keywords.

# knowledge_base = [
#     {
#         "id": "",
#         "title": "",
#         "keywords": [""],
#         "content": (
#             ""
#         ),
#     },
# ]
    
# Inclua documentos sobre disciplinas de sua preferência.

## 3. Selecione o contexto

Vamos usar uma estratégia transparente: normalizar os termos da pergunta e contar quantos deles aparecem em cada documento. Isso é suficiente para enxergar a etapa de seleção, mas não entende bem sinônimos.

In [ ]:
STOPWORDS = {
    "a", "o", "e", "de", "da", "do", "das", "dos", "em",
    "um", "uma", "para", "por", "com", "que", "na", "no",
    "as", "os", "qual", "quais", "como", "é", "são",
}

# Normalizamos o prompt do usuário, simulando a vetorização.
def normalize_terms(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return {term for term in text.split() if len(term) > 2 and term not in STOPWORDS}

# Ajudamos o agente a retornar com o contexto que mais possui os termos do prompt, simulando o retrieval.
def select_context(query, documents, max_docs=3, max_chars=1800):
    query_terms = normalize_terms(query)
    ranked = []

    for document in documents:
        searchable = " ".join([
            document["title"],
            document["content"],
            " ".join(document.get("keywords", [])),
        ])
        score = len(query_terms & normalize_terms(searchable))
        if score > 0:
            ranked.append({**document, "score": score})

    ranked.sort(key=lambda document: (-document["score"], document["id"]))

    selected = []
    used_chars = 0
    for document in ranked:
        cost = len(document["title"]) + len(document["content"])
        if selected and used_chars + cost > max_chars:
            continue
        selected.append(document)
        used_chars += cost
        if len(selected) >= max_docs:
            break

    return selected

## 4. Formate o contexto com fontes

O modelo precisa distinguir os dados externos da pergunta. Também queremos preservar as fontes para que a resposta consiga apontar de onde veio a informação.

In [ ]:
# O bloco contém ids de fonte para facilitar inspeção e atribuição. O modelo recebe dados externos claramente separados da pergunta.
def build_context(selected_documents):
    if not selected_documents:
        return "Nenhum documento relevante foi selecionado."

    blocks = ["<contexto>"]
    for document in selected_documents:
        blocks.append(
            f"[fonte: {document['id']} | relevância: {document['score']}]\n"
            f"{document['title']}\n{document['content']}"
        )
    blocks.append("</contexto>")
    return "\n\n".join(blocks)

## 5. Veja o que entrou no contexto

Antes de chamar a LLM, inspecione a decisão da aplicação. Context Engineering também é observabilidade: precisamos conseguir explicar qual informação foi enviada.

In [ ]:
query = "[DIGITE AQUI SUA PERGUNTA]"
selected = select_context(query, knowledge_base)
context = build_context(selected)

print(context)
print(f"\nDocumentos selecionados: {len(selected)}")
print(f"Caracteres de contexto: {len(context)}")

## 6. Monte a requisição com contexto

A instrução do sistema define como tratar as fontes. A mensagem do usuário contém o contexto selecionado e a pergunta atual. A aplicação continua responsável por montar tudo antes da chamada.

In [ ]:
# TODO: crie `messages_com_contexto` com uma mensagem system e uma user.
# Instrua o assistente a usar apenas o contexto fornecido, citar os ids
# das fontes quando possível e dizer quando o contexto não for suficiente.
# Envie a lista para o modelo e salve a resposta em `response_com_contexto`.

In [ ]:
# TODO: mostre o texto de `response_com_contexto` e os ids das fontes selecionadas.

print(response_com_contexto.choices[0].message.content)
print(f"\nFontes enviadas: {[document['id'] for document in selected]}")

In [ ]:
# Exemplo de instruções para o assistente:
# messages_com_contexto = [
#     {
#         "role": "system",
#         "content": (
#             "Você é um assistente de estudos. Use apenas as informações dentro de <contexto>. "
#             "Cite os ids das fontes quando fizer uma afirmação baseada nelas. "
#             "Se o contexto não for suficiente, diga isso claramente."
#         ),
#     },
#     {
#         "role": "user",
#         "content": f"{context}\n\nPergunta: {query}",
#     },
# ]

## 7. Compare sem contexto e com contexto

Escolha uma pergunta que dependa de uma informação da `knowledge_base`. Faça duas chamadas independentes. A primeira recebe apenas a pergunta; a segunda recebe o bloco selecionado. Compare especificidade, fontes e grau de confiança.

In [ ]:
# TODO: faça uma chamada sem contexto e salve em `response_sem_contexto`.
# Depois faça uma chamada com o contexto selecionado e salve em
# `response_com_contexto_2`. Imprima as duas respostas lado a lado.

## 8. Experimente o orçamento de contexto

Altere `max_chars` na função de seleção. Observe o que desaparece quando o orçamento diminui e por que a aplicação precisa escolher uma política explícita.

In [ ]:
for budget in [500, 1800]:
    selected_for_budget = select_context(
        query, knowledge_base, max_docs=3, max_chars=budget
    )
    context_for_budget = build_context(selected_for_budget)
    print(f"orçamento={budget}: ids={[document['id'] for document in selected_for_budget]} | caracteres={len(context_for_budget)}")

## 9. Checkpoint da V2

A V2 está pronta quando:

- a base de documentos existe;
- a seleção ocorre antes da chamada;
- o limite de documentos e caracteres é explícito;
- o contexto preserva as fontes;
- você comparou respostas sem contexto e com contexto;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.
